<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 06 · Alpha Is Rare

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks/labs"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Loading the Monthly Data
The file data/hf_data.csv contains monthly returns.


In [ ]:
from labs.lab06_alpha_is_rare import load_returns

In [ ]:
returns = load_returns()

In [ ]:
returns.head().round(4)

The first pass compares total return, annualized return, annualized
volatility, Sharpe ratio, and maximum drawdown.


In [ ]:
from labs.lab06_alpha_is_rare import comparison_summary

In [ ]:
summary = comparison_summary(returns)

In [ ]:
display = summary.copy()

In [ ]:
display["portfolio"] = ["HF", "SPY"]

In [ ]:
cols = [
    "portfolio",
    "total_return",
    "annualized_return",
    "annualized_vol",
    "max_drawdown",
]

In [ ]:
display[cols].round(3)

The next step is to ask how much of the hedge fund index return can
be explained by equity-market exposure.


In [ ]:
from labs.lab06_alpha_is_rare import alpha_summary

In [ ]:
alpha = alpha_summary(returns)

In [ ]:
alpha.round(3)

## Looking by Calendar Year
Aggregating monthly returns by calendar year helps identify when the hedge
fund index looked relatively strong and when it lagged badly.


In [ ]:
from labs.lab06_alpha_is_rare import yearly_returns

In [ ]:
yearly_returns(returns).round(3)

## Testing a Volatility-Scaling Story
One common defense of lower-return strategies is that they use less risk.


In [ ]:
from labs.lab06_alpha_is_rare import scenario_summary

In [ ]:
scenarios = scenario_summary(returns)

In [ ]:
display = scenarios.copy()

In [ ]:
display["portfolio"] = ["HF", "SPY", "60/40", "Levered HF"]

In [ ]:
cols = [
    "portfolio",
    "annualized_return",
    "annualized_vol",
    "max_drawdown",
    "beta_to_SPY",
]

In [ ]:
display[cols].round(3)

## Figure Generation (Optional)
Run the lab figure scripts under `code/figures/` to regenerate the PNG files
under `assets/figures/`.


In [ ]:
# Figure generation code adapted from `code/figures/lab06_cumulative_wealth.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from labs.lab06_alpha_is_rare import cumulative_wealth
from labs.lab06_alpha_is_rare import load_returns

def main() -> None:
    """Generate cumulative wealth paths."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    returns = load_returns()
    wealth = cumulative_wealth(returns)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(
        wealth.index,
        wealth["HF_INDEX"],
        linewidth=1.8,
        label="Hedge fund index",
    )
    ax.plot(wealth.index, wealth["SPY"], linewidth=1.8, label="SPY")
    ax.set_title("Cumulative wealth: hedge fund index vs SPY")
    ax.set_ylabel("Growth of 1.0")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="upper left")

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab06_risk_return_tradeoff.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

ROOT = PROJECT_ROOT

from labs.lab06_alpha_is_rare import load_returns
from labs.lab06_alpha_is_rare import scenario_summary

def main() -> None:
    """Generate a compact scenario comparison figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    stats = scenario_summary(load_returns()).set_index("portfolio")
    stats = stats.loc[
        ["Hedge fund index", "SPY", "60/40 SPY-IEF", "Levered hedge fund"]
    ]

    fig, axes = plt.subplots(1, 3, figsize=(8.4, 3.4), sharex=True)
    columns = [
        ("annualized_return", "Annualized return"),
        ("annualized_vol", "Annualized volatility"),
        ("max_drawdown", "Maximum drawdown"),
    ]
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    x = np.arange(len(stats.index))
    labels = ["HF", "SPY", "60/40", "Levered HF"]

    for ax, (column, title) in zip(axes, columns):
        values = stats[column].to_numpy()
        ax.bar(x, values, color=colors)
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)
        if column == "max_drawdown":
            ax.axhline(0.0, color="black", linewidth=0.8)

    fig.tight_layout()

main()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
